# BreastDCEDL ViT — pCR Prediction from DCE-MRI

End-to-end fine-tuning notebook for Google Colab (A100).

1. Upload the repository zip and extract
2. Download data from Zenodo (record 18114231)
3. Train ViT with two-phase LLRD, logged to Weights & Biases
4. Evaluate with subtype-stratified metrics

**Runtime:** GPU -> A100 (Runtime -> Change runtime type -> A100)

## 1. Upload Code & Install

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f"Uploaded: {zip_name}")

In [ ]:
import zipfile, shutil

WORK_DIR = "/content/algiers"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/_tmp")

extracted = os.listdir("/content/_tmp")
if len(extracted) == 1 and os.path.isdir(f"/content/_tmp/{extracted[0]}"):
    shutil.move(f"/content/_tmp/{extracted[0]}", WORK_DIR)
else:
    shutil.move("/content/_tmp", WORK_DIR)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. W&B Login

In [ ]:
import wandb
wandb.login()

## 3. Download Data from Zenodo

Record: [zenodo.org/records/18114231](https://zenodo.org/records/18114231)

| File | Size |
|------|------|
| `BreastDCEDL_ISPY2_min_crop.tar.gz` | 13.6 GB |
| `BreastDCEDL_DUKE_min_crop.tar.gz` | 8.6 GB |
| `BreastDCEDL_ISPY1_min_crop.tar.gz` | 1.2 GB |
| `BreastDCEDL_demo_data.tar.gz` | 55 MB |
| `BreastDCEDL_metadata_min_crop.csv` | 340 KB |

Choose which subsets to download below. For a quick test use `["demo", "metadata"]`.
For benchmarking use `["ispy2", "metadata"]` or all three cohorts.

In [ ]:
import sys
sys.path.insert(0, os.getcwd())

from src.data.zenodo import download_zenodo

DATA_ROOT = "/content/data"

# Quick test:
# download_zenodo(DATA_ROOT, subsets=["demo", "metadata"])

# Full benchmark (I-SPY2 + Duke + I-SPY1):
download_zenodo(DATA_ROOT, subsets=["ispy2", "duke", "ispy1", "metadata"])

## 4. Configure Paths

In [ ]:
import yaml

with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

# Point to Zenodo data
cfg["data"]["nifti"] = {
    "spy1": f"{DATA_ROOT}/BreastDCEDL_ISPY1_min_crop/spt1_dce",
    "spy2": f"{DATA_ROOT}/BreastDCEDL_ISPY2_min_crop/dce",
    "duke": f"{DATA_ROOT}/BreastDCEDL_DUKE_min_crop/duke_dce",
}
cfg["data"]["masks"] = {
    "spy1": f"{DATA_ROOT}/BreastDCEDL_ISPY1_min_crop/spy1_mask",
    "spy2": f"{DATA_ROOT}/BreastDCEDL_ISPY2_min_crop/mask",
    "duke": f"{DATA_ROOT}/BreastDCEDL_DUKE_min_crop/duke_mask",
}

# Use the Zenodo metadata CSV if present, else fall back to repo copy
zenodo_meta = f"{DATA_ROOT}/BreastDCEDL_metadata_min_crop.csv"
repo_meta = os.path.abspath("BreastDCEDL_metadata_min_crop.csv")
if os.path.isfile(zenodo_meta):
    cfg["data"]["combined_metadata"] = zenodo_meta
elif os.path.isfile(repo_meta):
    cfg["data"]["combined_metadata"] = repo_meta
elif os.path.isfile("BreastDCEDL_metadata.csv"):
    cfg["data"]["combined_metadata"] = os.path.abspath("BreastDCEDL_metadata.csv")
else:
    cfg["data"]["combined_metadata"] = None
print(f"Metadata: {cfg['data']['combined_metadata']}")

# A100 training settings
cfg["training"]["batch_size"] = 32
cfg["training"]["num_workers"] = 4
cfg["training"]["num_epochs"] = 30
cfg["checkpoint_dir"] = "checkpoints"

# W&B
cfg["wandb"] = {
    "enabled": True,
    "project": "breastdcedl-vit",
    "entity": None,
    "tags": ["vit", "pcr", "dce-mri", "a100"],
}

with open("configs/colab.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print("Saved configs/colab.yaml")

In [ ]:
# Verify extracted directory structure
for cohort, nifti_dir in cfg["data"]["nifti"].items():
    exists = os.path.isdir(nifti_dir)
    n = len(os.listdir(nifti_dir)) if exists else 0
    print(f"  {cohort}: {nifti_dir}  {'OK' if exists else 'MISSING'}  ({n} files)")

## 5. Verify Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.preprocessing import setup_paths, load_acquisitions, fuse_rgb_slice, select_timepoints, _cohort_from_pid

setup_paths(
    nifti_dirs=cfg["data"]["nifti"],
    mask_dirs=cfg["data"]["masks"],
)

meta_path = cfg["data"]["combined_metadata"]
df = pd.read_csv(meta_path)
print(f"Metadata: {len(df)} patients")
print(f"pCR rate: {df['pCR'].dropna().mean():.1%}")
if "dataset" in df.columns:
    print(f"Cohorts:  {df['dataset'].value_counts().to_dict()}")

In [ ]:
sample = df.dropna(subset=["pCR"]).sample(4, random_state=42)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (_, row) in zip(axes, sample.iterrows()):
    pid = row["pid"]
    acqs = load_acquisitions(pid)
    if acqs is None or len(acqs) < 2:
        ax.set_title(f"{pid}\n(no data)"); continue
    cohort = _cohort_from_pid(pid)
    pre, early, late = select_timepoints(acqs, cohort)
    mid_z = pre.shape[2] // 2
    rgb = fuse_rgb_slice(pre[:, :, mid_z], early[:, :, mid_z], late[:, :, mid_z])
    ax.imshow(rgb)
    pcr_label = "pCR" if row["pCR"] == 1 else "non-pCR"
    ax.set_title(f"{pid[:20]}...\n{pcr_label} | {cohort}")
    ax.axis("off")

plt.suptitle("RGB Fusion: Pre (R) / Early-Post (G) / Late-Post (B)", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Train

- **Phase 1** (5 epochs): Frozen backbone, head-only warmup
- **Phase 2** (25 epochs): Full fine-tune with LLRD (0.85)
- Early stopping on patient-level AUC (patience=10)
- All metrics logged to W&B

In [ ]:
from src.data.dataset import BreastDCEDataset, build_transforms
from src.data.splits import load_and_split
from src.models.vit import BreastDCEViT
from src.models.losses import FocalLoss, build_class_weights
from src.training.trainer import Trainer
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda")

dcfg = cfg["data"]
label_col = dcfg["label_col"]

train_df, val_df, test_df = load_and_split(
    dcfg["combined_metadata"], label_col=label_col,
)
print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print(f"Train pCR rate: {train_df[label_col].mean():.1%}")

In [ ]:
n_slices = dcfg.get("n_slices", 8)
crop_size = dcfg.get("crop_size", 224)
tcfg = cfg["training"]

train_ds = BreastDCEDataset(
    train_df, label_col=label_col, crop_size=crop_size,
    n_slices=n_slices,
    transform=build_transforms(tcfg.get("augmentation", {}), is_train=True),
)
val_ds = BreastDCEDataset(
    val_df, label_col=label_col, crop_size=crop_size,
    n_slices=n_slices,
    transform=build_transforms({}, is_train=False),
)

train_loader = DataLoader(
    train_ds, batch_size=tcfg["batch_size"], shuffle=True,
    num_workers=tcfg["num_workers"], pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=tcfg["batch_size"], shuffle=False,
    num_workers=tcfg["num_workers"], pin_memory=True,
)

print(f"Effective batch: {tcfg['batch_size'] * tcfg['accum_steps']}")
print(f"Train: {len(train_ds)} slices  Val: {len(val_ds)} slices")

In [ ]:
mcfg = cfg["model"]
model = BreastDCEViT(
    backbone=mcfg["backbone"],
    num_classes=mcfg["num_classes"],
    dropout=mcfg["dropout"],
).to(device)

print(f"Model: {mcfg['backbone']}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

weights = build_class_weights(train_df[label_col].tolist(), mcfg["num_classes"]).to(device)
criterion = FocalLoss(gamma=2.0, weight=weights)
print(f"Class weights: {weights.cpu().tolist()}")

In [ ]:
# Init W&B run
wandb.init(
    project=cfg["wandb"]["project"],
    entity=cfg["wandb"].get("entity"),
    tags=cfg["wandb"].get("tags", []),
    config={
        "model": cfg["model"],
        "training": cfg["training"],
        "data": {k: v for k, v in cfg["data"].items()
                 if k in ("label_col", "crop_size", "n_slices")},
        "train_patients": len(train_df),
        "val_patients": len(val_df),
        "test_patients": len(test_df),
        "pcr_rate": float(train_df[label_col].mean()),
    },
)
print(f"W&B run: {wandb.run.url}")

In [ ]:
trainer = Trainer(model, train_loader, val_loader, val_df, cfg, device)
best_auc = trainer.train(criterion)

## 7. Training Curves

In [ ]:
import json

with open(f"{cfg['checkpoint_dir']}/history.json") as f:
    history = json.load(f)

epochs = [e["epoch"] for e in history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, [e["train_loss"] for e in history], label="Train")
axes[0].plot(epochs, [e["val_loss"] for e in history], label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(epochs, [e["train_acc"] for e in history], label="Train")
axes[1].plot(epochs, [e["accuracy"] for e in history], label="Val (patient)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy"); axes[1].legend()

axes[2].plot(epochs, [e["auc"] for e in history], label="Val AUC", color="tab:green")
axes[2].axhline(y=0.72, color="red", linestyle="--", alpha=0.7, label="Paper overall (0.72)")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("AUC")
axes[2].set_title("Patient-Level AUC"); axes[2].legend()

plt.tight_layout()
plt.savefig(f"{cfg['checkpoint_dir']}/training_curves.png", dpi=150)
wandb.log({"training_curves": wandb.Image(fig)})
plt.show()

## 8. Evaluate on Test Set

In [ ]:
from torch.cuda.amp import autocast
from src.evaluation.metrics import (
    compute_metrics, evaluate_by_subtype,
    plot_roc, plot_confusion_matrix,
)

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

best_path = f"{cfg['checkpoint_dir']}/best.pth"
eval_model = BreastDCEViT(
    backbone=mcfg["backbone"],
    num_classes=mcfg["num_classes"],
    dropout=0.0,
).to(device)
eval_model.load_state_dict(torch.load(best_path, map_location=device))
eval_model.eval()
print(f"Loaded {best_path}")

In [ ]:
test_ds = BreastDCEDataset(
    test_df, label_col=label_col, crop_size=crop_size,
    n_slices=n_slices,
    transform=build_transforms({}, is_train=False),
)
test_loader = DataLoader(
    test_ds, batch_size=tcfg["batch_size"], shuffle=False,
    num_workers=tcfg["num_workers"], pin_memory=True,
)

all_logits, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        with autocast():
            logits = eval_model(images)
        all_logits.append(logits.cpu().float())
        all_labels.append(labels)

all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

n_patients = len(test_df)
logits_3d = all_logits[:n_patients * n_slices].view(n_patients, n_slices, -1)
labels_2d = all_labels[:n_patients * n_slices].view(n_patients, n_slices)

pooled = logits_3d.mean(dim=1)
patient_labels = labels_2d[:, 0].numpy()
probs = torch.softmax(pooled, dim=1)[:, 1].numpy()
preds = pooled.argmax(dim=1).numpy()

print(f"Test set: {n_patients} patients")

In [ ]:
overall = compute_metrics(patient_labels, probs, preds)
print("\n=== Overall Test Results ===")
for k, v in overall.items():
    print(f"  {k:15s}: {v:.4f}")

print("\n=== Paper Baseline (Table 2, Overall Test) ===")
print("  AUC:            0.72")
print("  Accuracy:       0.75")
print("  Sensitivity:    0.27")
print("  Specificity:    0.95")

# Log test metrics to W&B
wandb.run.summary.update({f"test/{k}": v for k, v in overall.items()})

In [ ]:
subtypes = cfg.get("evaluation", {}).get("subtypes", {})
sub_df = evaluate_by_subtype(
    test_df, probs, preds,
    label_col=label_col, subtypes=subtypes,
)
print("\n=== Subtype Breakdown ===")
print(sub_df.to_string(index=False, float_format="%.3f"))
sub_df.to_csv(f"{RESULTS_DIR}/subtype_results.csv", index=False)
wandb.log({"test/subtype_results": wandb.Table(dataframe=sub_df)})

In [ ]:
if "pid" in test_df.columns:
    cohort_results = []
    for name, pattern in [("Duke", "Breast_MRI"), ("I-SPY1", "SPY1"), ("I-SPY2", "SPY2")]:
        mask = test_df["pid"].str.contains(pattern, na=False).values
        if mask.sum() < 3:
            continue
        m = compute_metrics(patient_labels[mask], probs[mask], preds[mask])
        cohort_results.append({"Cohort": name, "N": int(mask.sum()), **m})
    if cohort_results:
        cohort_df = pd.DataFrame(cohort_results)
        print("\n=== Per-Cohort Breakdown ===")
        print(cohort_df.to_string(index=False, float_format="%.3f"))
        wandb.log({"test/cohort_results": wandb.Table(dataframe=cohort_df)})

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix as cm_fn
import seaborn as sns

# ROC
fpr, tpr, _ = roc_curve(patient_labels, probs)
auc_val = roc_auc_score(patient_labels, probs)

fig_roc, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc_val:.3f}")
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("Test Set ROC"); ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
fig_roc.savefig(f"{RESULTS_DIR}/roc_curve.png", dpi=150)
wandb.log({"test/roc_curve": wandb.Image(fig_roc)})
plt.show()

# Confusion matrix
cm = cm_fn(patient_labels, preds)
fig_cm, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-pCR", "pCR"], yticklabels=["Non-pCR", "pCR"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix")
plt.tight_layout()
fig_cm.savefig(f"{RESULTS_DIR}/confusion_matrix.png", dpi=150)
wandb.log({"test/confusion_matrix": wandb.Image(fig_cm)})
plt.show()

## 9. Save & Download

In [ ]:
pred_df = test_df[["pid"]].copy() if "pid" in test_df.columns else pd.DataFrame(index=range(n_patients))
pred_df["y_true"] = patient_labels
pred_df["y_prob"] = probs
pred_df["y_pred"] = preds
pred_df.to_csv(f"{RESULTS_DIR}/predictions.csv", index=False)

# Log predictions as W&B artifact
artifact = wandb.Artifact("test-predictions", type="predictions")
artifact.add_dir(RESULTS_DIR)
wandb.log_artifact(artifact)

wandb.finish()
print(f"Results saved to {RESULTS_DIR}/")
!ls -la {RESULTS_DIR}/

In [ ]:
!zip -r /content/breastdcedl_results.zip checkpoints/best.pth checkpoints/history.json results/
files.download("/content/breastdcedl_results.zip")